In [12]:
import pandas as pd
from sqlalchemy import text

from data_access.connection import engine
from data_access.config import project_path


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

## Создание DF на основе данных при инициализации задачи (то есть, момент, когда задача была создана и была заполнена информация о ее приоритете и т.д.)

Проблемой оказалось то, что приоритет на задачу могли установить не в момент ее инициализации, а на последующей итерации

In [13]:
query_path = project_path.query("training_dataset.sql")
query = text(query_path.read_text(encoding="utf-8"))

issue_df = pd.read_sql_query(query, engine)
issue_df

,ID,Project_ID,Creation_Date,Description_Issue_Length,Resolution_Date,Duration_Days,Status,Initial_Priority,Initial_Type,Description_Project_Length,Comment_Count,Comment_Total_Length,Change_Count
0,85,1,2016-02-24 10:51:02,249,2016-12-23 14:49:45,303.1658,Done,Minor,Bug,146,1,791.0,13
1,86,1,2016-02-23 18:31:38,125,2016-12-23 14:49:26,303.8457,Done,Minor,Improvement,146,0,0.0,13
2,89,1,2016-02-22 15:27:31,384,2016-02-22 18:41:31,0.1347,Done,Minor,Story,146,0,0.0,11
3,90,1,2016-02-17 14:28:31,18,2016-02-21 02:28:25,3.4999,Done,Minor,Bug,146,0,0.0,10
4,91,1,2016-02-16 14:59:06,579,2016-02-19 20:04:19,3.2120,Done,Minor,Improvement,146,0,0.0,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...
251867,509031,33,2009-04-09 09:15:20,68,2009-07-14 16:27:28,96.3001,Closed,Major,Improvement,130,0,0.0,11
251868,509032,33,2009-04-09 09:14:27,192,2010-02-25 21:42:33,322.5195,Closed,Major,Improvement,130,3,216.0,19
251869,509033,33,2009-04-09 09:02:06,107,2009-05-27 14:08:21,48.2127,Closed,Minor - P4,Bug,130,6,1178.0,17
251870,509034,33,2009-04-09 08:59:26,0,2009-08-25 11:29:57,138.1045,Closed,Major - P3,Improvement,130,1,73.0,14


## Проверка структуры итогового датасета

In [14]:
structure_summary = pd.Series({
    "row_count": len(issue_df),
    "unique_id_count": issue_df["ID"].nunique(),
    "duplicate_id_count": issue_df["ID"].duplicated().sum(),
})

structure_summary

row_count             251872
unique_id_count       251872
duplicate_id_count         0
dtype: int64

In [15]:
missing_values = issue_df.isna().sum().sort_values(ascending=False)

missing_values

ID                            0
Project_ID                    0
Creation_Date                 0
Description_Issue_Length      0
Resolution_Date               0
Duration_Days                 0
Status                        0
Initial_Priority              0
Initial_Type                  0
Description_Project_Length    0
Comment_Count                 0
Comment_Total_Length          0
Change_Count                  0
dtype: int64

In [16]:
duration_quality = pd.Series({
    "negative_duration_count": issue_df["Duration_Days"].lt(0).sum(),
    "zero_duration_count": issue_df["Duration_Days"].eq(0).sum(),
    "missing_duration_count": issue_df["Duration_Days"].isna().sum(),
})

duration_quality

negative_duration_count    0
zero_duration_count        0
missing_duration_count     0
dtype: int64

In [17]:
issue_df["Duration_Days"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

count    251872.000000
mean         50.663282
std          79.344728
min           0.003500
50%          13.905600
90%         170.002690
95%         248.936480
99%         338.105492
max         364.999800
Name: Duration_Days, dtype: float64

In [18]:
category_summary = issue_df[[
    "Project_ID",
    "Initial_Type",
    "Initial_Priority",
]].nunique()

category_summary

Project_ID          39
Initial_Type        69
Initial_Priority    19
dtype: int64

In [19]:
issue_df["Initial_Priority"].value_counts(dropna=False)

Initial_Priority
Minor             47162
Major - P3        46577
Unknown           33229
Medium            31027
Low               27229
Major             22121
None              12204
High               9192
To be reviewed     7294
Critical           4151
Minor - P4         3242
Blocker            2441
Critical - P2      1503
Highest            1501
Trivial            1399
Trivial - P5        608
Blocker - P1        573
Undefined           306
Lowest              113
Name: count, dtype: int64

In [20]:
issue_df["Initial_Type"].value_counts(dropna=False)

Initial_Type
Bug               132457
Improvement        32913
Task               25204
Story              24412
New Feature        11564
                   ...  
Action Item            1
IT Help                1
Module Request         1
Workload               1
Change                 1
Name: count, Length: 69, dtype: int64

In [21]:
issue_df["Status"].value_counts(dropna=False)

Status
Closed                    214015
Done                       23916
Resolved                    8386
Complete                    3720
Won't Fix                    959
Invalid                      594
Open                         221
Planned Development           36
Reopened                       7
Gathering Interest             5
In Progress                    4
Pull Request Submitted         3
In Review                      2
Gathering Impact               1
Published                      1
Needs Triage                   1
Automation                     1
Name: count, dtype: int64

## Хронологическое разделение данных

In [22]:
issue_df = issue_df.sort_values(
    ["Creation_Date", "ID"]
).reset_index(drop=True)

train_end = int(len(issue_df) * 0.70)
validation_end = int(len(issue_df) * 0.85)

train_df = issue_df.iloc[:train_end].copy()
validation_df = issue_df.iloc[train_end:validation_end].copy()
test_df = issue_df.iloc[validation_end:].copy()

In [23]:
split_summary = pd.DataFrame({
    "train": {
        "row_count": len(train_df),
        "creation_date_min": train_df["Creation_Date"].min(),
        "creation_date_max": train_df["Creation_Date"].max(),
    },
    "validation": {
        "row_count": len(validation_df),
        "creation_date_min": validation_df["Creation_Date"].min(),
        "creation_date_max": validation_df["Creation_Date"].max(),
    },
    "test": {
        "row_count": len(test_df),
        "creation_date_min": test_df["Creation_Date"].min(),
        "creation_date_max": test_df["Creation_Date"].max(),
    },
}).T

split_summary

,row_count,creation_date_min,creation_date_max
train,176310,2004-01-02 14:51:51,2017-04-24 20:27:42
validation,37781,2017-04-24 21:05:52,2018-11-08 15:21:36
test,37781,2018-11-08 15:43:48,2020-10-22 15:16:24


In [24]:
duration_distribution = pd.DataFrame({
    "train": train_df["Duration_Days"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ),
    "validation": validation_df["Duration_Days"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ),
    "test": test_df["Duration_Days"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ),
}).T

duration_distribution

,count,mean,std,min,50%,90%,95%,99%,max
train,176310.0,52.087119,82.084503,0.0035,13.0486,181.60962,257.912765,341.075669,364.9998
validation,37781.0,50.246869,78.516613,0.0035,14.5350,166.93340,246.062000,339.521920,364.9874
test,37781.0,44.435173,65.682734,0.0035,16.8415,128.46800,191.898300,313.368180,364.9152
